In [1]:
import pandas as pd

EXPORTS = "outputs/dataset_exports"

In [2]:
# Profile-level dataset (one row per tutor). Already includes derived target_subjects, review aggregates, etc.
profiles = pd.read_excel(f"{EXPORTS}/target_ege_dataset.xlsx")
print(profiles.shape)
profiles.head()

(27052, 52)


             id  ... is_target_ege_profile
0       AaaAA40  ...                  True
1   AbadzhidiSP  ...                  True
2   AbakarovaTS  ...                  True
3    AbakarovIS  ...                  True
4  AbakumovAA17  ...                  True

[5 rows x 52 columns]

In [3]:
import re

# Long-format prices: one row per (profile, target subject).
prices_long = pd.read_csv(f"{EXPORTS}/target_ege_prices_long.csv")

# Source mixes units in price_text (60 мин. / мес. / нед. / etc.). Keep only per-hour rates so aggregates are comparable.
def _unit(t):
    if not isinstance(t, str):
        return None
    m = re.search(r"₽/(\S+(?:\s\S+)?)", t)
    return m.group(1) if m else None

prices_long["unit"] = prices_long["price_text"].apply(_unit)
unit_counts = prices_long["unit"].value_counts(dropna=False)
print("unit counts:")
print(unit_counts)

hourly_mask = prices_long["unit"].eq("60 мин.")
# Drop obvious data-entry garbage: < 100 ₽/h or > 100k ₽/h (legit max in distribution sits ~10k).
plausible_mask = prices_long["price_min"].between(100, 100_000)
prices_clean = prices_long[hourly_mask & plausible_mask].copy()
print(f"\nkept {len(prices_clean):,} of {len(prices_long):,} price rows after unit + outlier filter")
prices_clean.head()

unit counts:
unit
60 мин.    31199
мес.          23
мин.          10
нед.           3
6 ч            1
день           1
Name: count, dtype: int64

kept 31,194 of 31,237 price rows after unit + outlier filter


             id                     service_key  ... target_subjects     unit
0   AbadzhidiSP     /repetitor/ege/ege_english/  ...     ege_english  60 мин.
1   AbakarovaTS     /repetitor/ege/ege_physics/  ...     ege_physics  60 мин.
2   AbakarovaTS  /repetitor/ege/ege_matematika/  ...        ege_math  60 мин.
3  AbakumovAA17  /repetitor/ege/ege_matematika/  ...        ege_math  60 мин.
4   AbakumovaSA  /repetitor/ege/ege_matematika/  ...        ege_math  60 мин.

[5 rows x 8 columns]

In [4]:
# Collapse hourly prices to one row per profile so the merge stays profile-level.
price_agg = (
    prices_clean.groupby("id")
    .agg(
        price_min=("price_min", "min"),
        price_max=("price_max", "max"),
        price_avg=("price_avg", "mean"),
        price_subjects_count=("target_subjects", "nunique"),
    )
    .reset_index()
)
print(price_agg.shape)
price_agg.head()

(19695, 5)


             id  price_min  price_max    price_avg  price_subjects_count
0   AbadzhidiSP       2000        NaN  2000.000000                     1
1   AbakarovaTS       2000     5000.0  3125.000000                     2
2  AbakumovAA17       2000        NaN  2000.000000                     1
3   AbakumovPV4       1800     3000.0  2383.333333                     2
4   AbakumovaSA       1700        NaN  1700.000000                     1

In [5]:
# Left-merge so every profile is kept; profiles with no price rows get NaN price stats and 0 subjects.
df = profiles.merge(price_agg, how="left", on="id")
df["price_subjects_count"] = df["price_subjects_count"].fillna(0).astype(int)

assert len(df) == len(profiles), "merge changed row count — duplicates in price_agg"
print(df.shape)

(27052, 56)


In [6]:
# Convert JSON-array profile fields to flat text (entries joined with " | ") so they survive in CSV form.
import json as _json

def _flatten_json_list(val):
    if not isinstance(val, str) or not val:
        return None
    try:
        items = _json.loads(val)
    except _json.JSONDecodeError:
        return val
    if not isinstance(items, list) or not items:
        return None
    return " | ".join(str(x).strip() for x in items if x)

flatten_map = [
    ("education_json", "education"),
    ("experience_json", "experience"),
    ("achievements_json", "achievements"),
    ("additional_info_json", "additional_info"),
]
for src_col, new_col in flatten_map:
    if src_col in df.columns:
        df[new_col] = df[src_col].apply(_flatten_json_list)

# Drop columns that bloat the table without helping EDA: original JSON blobs (now flattened above),
# the prices_json blob, parser/runtime metadata, and the always-True target flag.
drop_cols = [
    "education_json",
    "experience_json",
    "achievements_json",
    "additional_info_json",
    "prices_json",
    "source_url",
    "parsed_at",
    "data_source",
    "runtime_seconds",
    "reviews_data_source",
    "reviews_parsed_at",
    "reviews_runtime_seconds",
    "reviews_count_reviews_file",
    "filtered_count",
    "reviews_loaded",
    "reviews_1_count_reviews_file",
    "reviews_2_count_reviews_file",
    "reviews_3_count_reviews_file",
    "reviews_4_count_reviews_file",
    "reviews_5_count_reviews_file",
    "reviews_stars_sum",
    "stars_sum_equals_reviews_count",
    "is_target_ege_profile",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)

(27052, 37)


In [7]:
# Light type cleanup so EDA tooling treats columns sensibly.
# client_visit is free-text describing visit area when offered; convert presence to a boolean and drop the text.
if "client_visit" in df.columns:
    df["client_visit_available"] = df["client_visit"].notna()
    df = df.drop(columns=["client_visit"])

for col in ["passport_verified", "remote_available", "is_target_ege_profile", "client_visit_available"]:
    if col in df.columns:
        df[col] = df[col].astype("boolean")

if "gender_binary" in df.columns:
    df["gender_binary"] = df["gender_binary"].astype("Int8")

df.dtypes

id                                                str
name                                              str
gender_raw                                        str
gender_binary                                    Int8
reviews_count                                   int64
rating_avg                                    float64
reviews_1_count                                 int64
reviews_2_count                                 int64
reviews_3_count                                 int64
reviews_4_count                                 int64
reviews_5_count                                 int64
passport_verified                             boolean
remote_available                              boolean
about                                             str
about_len                                       int64
education_count                                 int64
experience_count                                int64
achievements_count                              int64
additional_info_count       

In [8]:
# Extract years of experience: max integer N from "N лет / N год / N года" mentions in experience text.
# Cap at < 80 so 4-digit year references like "с 2007 года" can't masquerade as duration.
_years_re = re.compile(r"(\d+)\s+(?:лет|года?)\b")
_PLAUSIBLE_MAX_YEARS = 80

def _max_years(text):
    if not isinstance(text, str):
        return None
    candidates = [int(m) for m in _years_re.findall(text) if int(m) < _PLAUSIBLE_MAX_YEARS]
    return max(candidates) if candidates else None

df["experience_years"] = df["experience"].apply(_max_years).astype("Int16")
print(df["experience_years"].describe())

count      23576.0
mean     10.008992
std        9.47555
min            1.0
25%            3.0
50%            6.0
75%           14.0
max           62.0
Name: experience_years, dtype: Float64


In [9]:
# Time on Profi.ru — parsed from "На сервисе с {месяц} {год} г" in the experience text.
# Compute from the date itself (not the parenthesized hint) so freshly-joined tutors get a real value.
RU_MONTHS = {
    "января": 1, "февраля": 2, "марта": 3, "апреля": 4, "мая": 5, "июня": 6,
    "июля": 7, "августа": 8, "сентября": 9, "октября": 10, "ноября": 11, "декабря": 12,
}
_on_site_re = re.compile(r"На сервисе с\s+([А-Яа-яё]+)\s+(\d{4})\s*г")
TODAY = pd.Timestamp("2026-05-07")

def _years_on_site(text):
    if not isinstance(text, str):
        return None
    m = _on_site_re.search(text)
    if not m:
        return None
    month = RU_MONTHS.get(m.group(1).lower())
    if not month:
        return None
    start = pd.Timestamp(year=int(m.group(2)), month=month, day=1)
    months = (TODAY.year - start.year) * 12 + (TODAY.month - start.month)
    return round(max(months, 0) / 12, 1)

df["years_on_site"] = df["experience"].apply(_years_on_site).astype("Float32")
print(df["years_on_site"].describe())
print("\nparsed for", df["years_on_site"].notna().sum(), "of", len(df), "profiles")

count     27052.0
mean     5.647283
std      4.788439
min           0.0
25%           1.5
50%           4.6
75%           8.7
max          20.5
Name: years_on_site, dtype: Float64

parsed for 27052 of 27052 profiles


In [10]:
# Top-university flag: SQL LIKE-style substring match (case-sensitive) of curated abbreviations / canonical names
# against the education text. Substring is intentional — matches "МГУ" inside "МГУ им. М.В. Ломоносова" etc.
TOP_UNIVERSITIES = [
    "ВШЭ", "МГУ", "МФТИ", "ИТМО", "СПбГУ", "МИФИ", "МГТУ", "МИСИС", "РЭУ", "МГИМО",
    "Финансовый университет", "РАНХиГС", "НГУ", "РЭШ", "СПбПУ", "УрФУ", "Иннополис", "РУДН",
    "ЦУ", "Сеченов",
]
_uni_re = re.compile("|".join(re.escape(u) for u in TOP_UNIVERSITIES))

def _matched_unis(text):
    if not isinstance(text, str):
        return None
    hits = _uni_re.findall(text)
    return " | ".join(sorted(set(hits))) if hits else None

df["top_universities_matched"] = df["education"].apply(_matched_unis)
df["is_top_university"] = df["top_universities_matched"].notna().astype("boolean")
print("is_top_university counts:", df["is_top_university"].value_counts(dropna=False).to_dict())
print("top matched universities (top 15):")
print(df["top_universities_matched"].value_counts().head(15))

is_top_university counts: {np.False_: 19690, np.True_: 7362}
top matched universities (top 15):
top_universities_matched
МГУ                       2482
МГТУ                      1000
ВШЭ                        812
МИФИ                       521
МФТИ                       458
МГИМО                      339
РУДН                       292
Финансовый университет     264
РАНХиГС                    180
РЭУ                        142
МИСИС                      121
СПбГУ                       92
Сеченов                     85
НГУ                         69
ВШЭ | МГУ                   63
Name: count, dtype: int64


In [11]:
# Drop profiles with no review activity in the last 6 months. Cutoff is anchored to today (2026-05-07).
reviews = pd.read_csv(f"{EXPORTS}/target_ege_reviews_long.csv", usecols=["id", "date_timestamp"])
reviews["date"] = pd.to_datetime(reviews["date_timestamp"], unit="s", errors="coerce")
cutoff = pd.Timestamp("2026-05-07") - pd.DateOffset(months=6)

last_review = reviews.groupby("id")["date"].max()
recent_ids = set(last_review[last_review >= cutoff].index)

before = len(df)
df = df[df["id"].isin(recent_ids)].reset_index(drop=True)
print(f"recent-reviews filter (cutoff {cutoff.date()}): kept {len(df):,} of {before:,} profiles")

recent-reviews filter (cutoff 2025-11-07): kept 1,448 of 27,052 profiles


In [12]:
# Drop profiles with no gender — needed for any gender-based analysis (cannot impute meaningfully).
before = len(df)
df = df[df["gender_raw"].notna()].reset_index(drop=True)
print(f"gender-not-null filter: kept {len(df):,} of {before:,} profiles "
      f"(dropped {before - len(df)})")

gender-not-null filter: kept 1,413 of 1,448 profiles (dropped 35)


In [13]:
# Quick sanity summary before saving.
print("shape:", df.shape)
print("unique profiles:", df["id"].nunique())
print("\nmissing share (top 15):")
print(df.isna().mean().sort_values(ascending=False).head(15))
print("\nprice stats:")
print(df[["price_min", "price_max", "price_avg", "price_subjects_count"]].describe())

shape: (1413, 41)
unique profiles: 1413

missing share (top 15):
additional_info                      0.828025
top_universities_matched             0.702052
price_max                            0.607219
achievements                         0.522293
target_subjects_from_about           0.395612
price_avg                            0.200283
price_min                            0.200283
target_subjects_from_prices          0.199575
target_price_service_keys            0.199575
experience_years                     0.123850
education                            0.090587
about                                0.029724
target_review_service_count_names    0.000000
target_subjects_from_reviews         0.000000
target_review_service_names          0.000000
dtype: float64

price stats:
         price_min     price_max     price_avg  price_subjects_count
count  1130.000000    555.000000   1130.000000           1413.000000
mean   1930.769912   3881.117117   2495.731268              0.977353
std     8

In [14]:
out_path = f"{EXPORTS}/clean_ege_dataset.csv"
df.to_csv(out_path, index=False)
print("saved:", out_path)
df.head()

saved: outputs/dataset_exports/clean_ege_dataset.csv


                 id  ... is_top_university
0       AbakumovPV4  ...              True
1       AbalakinaNS  ...             False
2       AbalakovaSA  ...             False
3  AbdulzhalilovaAK  ...             False
4         AbduyevAO  ...              True

[5 rows x 41 columns]

In [15]:
# Per-(profile, subject) frame from the cleaned hourly prices, with subject-scoped column names
# so they don't collide with the profile-level price aggregates already in df.
subject_prices = prices_clean.rename(
    columns={
        "target_subjects": "subject",
        "service_key": "subject_service_key",
        "price_text": "subject_price_text",
        "price_min": "subject_price_min",
        "price_max": "subject_price_max",
        "price_avg": "subject_price_avg",
    }
)[
    [
        "id",
        "subject",
        "subject_service_key",
        "subject_price_text",
        "subject_price_min",
        "subject_price_max",
        "subject_price_avg",
    ]
]

# Inner join: only keep profiles that have at least one priced subject (otherwise no row to emit).
df_long = subject_prices.merge(df, how="inner", on="id")
print("long-format shape:", df_long.shape)
print("unique profiles:", df_long["id"].nunique())
print("rows per subject:")
print(df_long["subject"].value_counts())

long-format shape: (2092, 47)
unique profiles: 1130
rows per subject:
subject
ege_math           1287
ege_russian         254
ege_english         206
ege_physics         191
ege_informatics     154
Name: count, dtype: int64


In [16]:
long_path = f"{EXPORTS}/clean_ege_dataset_long.csv"
df_long.to_csv(long_path, index=False)
print("saved:", long_path)
df_long.head()

saved: outputs/dataset_exports/clean_ege_dataset_long.csv


            id      subject  ... top_universities_matched is_top_university
0  AbakumovPV4  ege_physics  ...                     МГТУ              True
1  AbakumovPV4     ege_math  ...                     МГТУ              True
2  AbakumovPV4     ege_math  ...                     МГТУ              True
3  AbalakinaNS  ege_english  ...                      NaN             False
4  AbalakovaSA  ege_russian  ...                      NaN             False

[5 rows x 47 columns]

In [17]:
len(df)

1413